In [1]:
import re

import os
os.environ['CUDA_VISIBLE_DEVICES'] = "4"



import torch
from PIL import Image, ImageDraw, ImageFont  # 导入绘制库
from transformers import AutoModel, AutoTokenizer, AutoProcessor


In [2]:
model_path = "LocateAnything-3B"
device  = "cuda"
dtype=torch.float16


In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
model = AutoModel.from_pretrained(
            model_path,
            torch_dtype=dtype,
            trust_remote_code=True,
        ).to(device).eval()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/home/ntz/anaconda3/lib/python3.12/site-packages/transformers/models/auto/image_processing_auto.py:647: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Qwen2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
model

LocateAnythingForConditionalGeneration(
  (vision_model): MoonVitPretrainedModel(
    (patch_embed): MoonVisionPatchEmbed(
      (proj): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14))
      (pos_emb): Learnable2DInterpPosEmb()
    )
    (encoder): MoonVitEncoder(
      (rope_2d): Rope2DPosEmb(dim=72, max_height=512, max_width=512, theta_base=10000)
      (blocks): ModuleList(
        (0-26): 27 x MoonVitEncoderLayer(
          (norm0): LayerNorm((1152,), eps=1e-05, elementwise_affine=True, bias=True)
          (norm1): LayerNorm((1152,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): MLP2(
            (fc0): Linear(in_features=1152, out_features=4304, bias=True)
            (fc1): Linear(in_features=4304, out_features=1152, bias=True)
            (activation): GELU(approximate='tanh')
          )
          (wqkv): Linear(in_features=1152, out_features=3456, bias=True)
          (wo): Linear(in_features=1152, out_features=1152, bias=True)
        )
      )
     

In [5]:
model.language_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152681, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm()
        (post_attention_layernorm): Qwen2RMSNorm()
      )
    )
    (norm): Qwen2RMSNorm()
  )
  (lm_

In [7]:
model.language_model.model.embed_tokens

Embedding(152681, 2048)

In [8]:
import torch
from safetensors.torch import save_file

# 1. 获取对应的 config 变量（如果你的 model.config 包含这些的话）
# 否则也可以直接硬编码，从你的打印信息看：vocab_size=152681, hidden_size=2048
vocab_size = model.language_model.config.vocab_size
hidden_size = model.language_model.config.hidden_size

# 2. 提取 embed_tokens 层的权重，并转移到 CPU 上
embed_weight = model.language_model.model.embed_tokens.weight.detach().cpu()

# 3. 将其保存为 safetensors 格式
save_dict = {"weight": embed_weight}
save_file(save_dict, "qwen2_embed_tokens.safetensors")

print(f"Embedding 权限已保存，形状为: {embed_weight.shape}") 
# 预期输出: 形状为: torch.Size([152681, 2048])

Embedding 权限已保存，形状为: torch.Size([152681, 2048])


In [ ]:
import torch
import os

# 假设 `model` 是你的完整多模态模型 (例如 Qwen2-VL)
# tokenizer 是你对应的分词器

output_path = "./locate_qwen2_model"
os.makedirs(output_path, exist_ok=True)

# 1. 直接调用 language_model 的 save_pretrained
# 参数 safe_serialization=True 会强制将权重保存为 safetensors 格式
model.language_model.save_pretrained(
    output_path,
    safe_serialization=True 
)

# 2. 【极其重要】记得将分词器也保存到同一个目录下
# 因为 vLLM 加载时需要 tokenizer_config.json 和 vocab 数据
tokenizer.save_pretrained(output_path)

print(f"语言模型已成功剥离并保存至: {output_path}")

In [ ]:
# run vllm in other cli/bash interface
#CUDA_VISIBLE_DEVICES=3 vllm serve locate_qwen2_model --tensor-parallel-size 1 --max-model-len 8192  --gpu-memory-utilization 0.7   --dtype half   --kv-cache-dtype auto   --block-size 16   --max-model-len 16384   --max-num-seqs 4   --max-cudagraph-capture-size 8   --attention-backend TRITON_ATTN  --enable-prompt-embeds

In [ ]:
### 这里开始

In [9]:
import torch
import torch.nn as nn
from safetensors.torch import load_file

# 1. 明确定义参数 (对应前文提取的参数)
vocab_size = 152681
hidden_size = 2048
# Qwen2 的 padding_idx 通常是不设置的，或者是配置文件里的 config.pad_token_id
padding_idx = None # 或者按需设置为你的 token ID，例如 151643

# 2. 独立实例化一个纯净的 nn.Embedding 层
standalone_embed_tokens = nn.Embedding(
    num_embeddings=vocab_size, 
    embedding_dim=hidden_size, 
    padding_idx=padding_idx
)

# 3. 加载刚才保存的 safetensors 权重
state_dict = load_file("qwen2_embed_tokens.safetensors")

# 因为 state_dict 里 key 为 "weight"，这与 nn.Embedding 内置的权重名称完美匹配
standalone_embed_tokens.load_state_dict(state_dict)

# ✅ 如果你需要它在 GPU 上运行，也可以移动过去
standalone_embed_tokens = standalone_embed_tokens.to("cuda")

print("加载成功！standalone_embed_tokens 已经可以独立工作了。")

加载成功！standalone_embed_tokens 已经可以独立工作了。


In [10]:
categories = ["person", "ballet heel", "bicycle"]
image = Image.open("DSC_1242.jpg").convert("RGB")

cats = "</c>".join(categories)
prompt = f"Locate all the instances that matches the following description: {cats}."

messages = [
    {"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ]}
]

text = processor.py_apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
images, videos = processor.process_vision_info(messages)
inputs = processor(
    text=[text], images=images, videos=videos, return_tensors="pt"
).to(device)

pixel_values = inputs["pixel_values"].to(dtype)
input_ids = inputs["input_ids"]

In [13]:
import numpy as np
# ==========================================================
# 3. 提取视觉特征提取并映射 (Vision Encoder + Mlp1)
# ==========================================================
# 检查是否包含 image_grid_hws
image_grid_hws = inputs.get("image_grid_hws", None) 

if isinstance(image_grid_hws, np.ndarray):
    image_grid_hws = torch.from_numpy(image_grid_hws).to(pixel_values.device, dtype=torch.int32)

with torch.no_grad():
    # 这里对应原本的 extract_feature 和 mlp1 逻辑
    vit_embeds = model.extract_feature(pixel_values, image_grid_hws)
    
    # 按照原模型处理列表/拼接的逻辑
    if isinstance(vit_embeds, list):
        vit_embeds = torch.cat(vit_embeds, dim=0)
    elif image_grid_hws is not None:
        vit_embeds = torch.cat(vit_embeds, dim=0)

    # 通过 MLP 投影层映射到 LLM 的维度 (hidden_size)
    vit_embeds = model.mlp1(vit_embeds)



In [15]:
# ==========================================================
# 4. 执行 image_processing：将 Image Embeds 与 Text Embeds 缝合
# ==========================================================
image_token_index = model.image_token_index

with torch.no_grad():
    # （1）先用原生语言模型的 vocab embedding 层把全部 input_ids 变成文字特征
    # input_embeds = model.language_model.get_input_embeddings()(input_ids)
    input_embeds = standalone_embed_tokens(input_ids).to(dtype)
    B, N, C = input_embeds.shape
    input_embeds = input_embeds.reshape(B * N, C)
    input_ids_flat = input_ids.reshape(B * N)
    
    # （2）定位 <image> token 的位置
    selected = (input_ids_flat == image_token_index)
    assert selected.sum() != 0, "Error: No image token index found in the input_ids!"
    
    # （3）替换特征：将计算出的 vit_embeds 填入对应的 token 槽位
    input_embeds[selected] = vit_embeds.reshape(-1, C).to(input_embeds.device)
    
    # 将形状变回 (B, N, C)
    input_embeds = input_embeds.reshape(B, N, C)



In [21]:
import pybase64
import io
from openai import OpenAI
def tensor2base64(x: torch.Tensor) -> str:
    with io.BytesIO() as buf:
        torch.save(x, buf)
        buf.seek(0)
        binary_data = buf.read()

    return pybase64.b64encode(binary_data).decode("utf-8")



In [28]:


# ==========================================================
# 5. vLLM 推理：调用 Completions API 
# ==========================================================
# 去掉 batch 维度，vLLM API 请求期望单条 embedding 序列的形状为 (seq_len, hidden_size)
prompt_embeds = input_embeds.squeeze(0).clone().detach() 
print(prompt_embeds.size())
# Base64 序列化
encoded_embeds = tensor2base64(prompt_embeds)

# 初始化 OpenAI 客户端
client = OpenAI(
    api_key="EMPTY",
    base_url="http://localhost:8000/v1",  # 确保你用 vllm serve 启动了独立出来的纯文本模型底座
)

# 纯粹的 AR 模式被转移到了远端生成
print("开始请求 vLLM 进行生成...")
completion = client.completions.create(
    model="locate_qwen2_model", # 这里填入你启动 vLLM 时的纯文本模型名字
    prompt=None,  # 留空，因为全部特征都已经通过 extra_body 传过去了
    max_tokens=2048,
    temperature=0.0,
    # 额外携带计算好的多模态特征矩阵s
    extra_body={"prompt_embeds": encoded_embeds, 
                "return_token_ids": True,
                "skip_special_tokens": False,
        "spaces_between_special_tokens": False},
)

print("\n--- vLLM 返回的输出结果 ---")
print(completion.choices[0].text)

torch.Size([673, 2048])
开始请求 vLLM 进行生成...

--- vLLM 返回的输出结果 ---
<ref>person</ref><box><155><15><871><951></box><box><464><34><915><964></box><ref>ballet heel</ref><box><250><682><422><946></box><box><364><670><497><950></box><box><548><824><699><964></box><box><705><821><842><952></box><ref>bicycle</ref><box>None</box>


In [30]:
import re
def parse_boxes(answer: str, image_width: int, image_height: int) -> list[dict]:
    """解析模型输出，提取检测框及其对应的标签（处理多box共享同一个ref的情况）。"""
    boxes = []
    # 改良正则：采用分支匹配，要么匹配 <ref>标签</ref>，要么匹配坐标框 <box><x1><y1><x2><y2></box>
    pattern = r"<ref>(.*?)</ref>|<box><(\d+)><(\d+)><(\d+)><(\d+)></box>"
    
    current_label = "object"  # 初始化默认标签

    # 遍历所有匹配项，顺序解析
    for m in re.finditer(pattern, answer):
        if m.group(1) is not None:
            # 匹配到了 <ref>标签</ref>，更新当前正在解析的种类
            current_label = m.group(1).strip()
        elif m.group(2) is not None:
            # 匹配到了包含数字的坐标 <box>，附加当前标签
            x1, y1, x2, y2 = [int(g) for g in m.groups()[1:]]
            boxes.append({
                "label": current_label,
                "x1": x1 / 1000 * image_width,
                "y1": y1 / 1000 * image_height,
                "x2": x2 / 1000 * image_width,
                "y2": y2 / 1000 * image_height,
            })
    return boxes

In [29]:
def draw_boxes(image: Image.Image, boxes: list[dict], width: int = 3) -> Image.Image:
    """在图片上画框并标注标签，不同类别使用不同颜色。"""
    from PIL import ImageDraw, ImageFont
    
    # copy 图像，防止修改原图
    draw_img = image.copy()
    draw = ImageDraw.Draw(draw_img)
    
    # 尝试加载默认字体，如果不可用则使用系统默认
    try:
        font = ImageFont.load_default()
    except IOError:
        font = None

    # 准备一个高对比度的颜色面板（可根据喜好在此处添加更多十六进制颜色或颜色单词）
    color_palette = [
        "#FF3838",  # 红色
        "#2C99A8",  # 湖蓝色
        "#FF701F",  # 橙色
        "#6473FF",  # 紫蓝色
        "#17C825",  # 绿色
        "#FF9D97",  # 粉红色
        "#9D20F5",  # 紫色
        "#EBEB00",  # 黄色
    ]
    
    # 用于记录 标签名 -> 颜色 的对应关系，保证同类的框颜色相同
    label2color = {}

    for box in boxes:
        coordinates = [box["x1"], box["y1"], box["x2"], box["y2"]]
        label = box['label']
        
        # 动态分配颜色：如果当前标签还没分配颜色，则从颜色面板中按顺序取一个
        if label not in label2color:
            color_index = len(label2color) % len(color_palette)
            label2color[label] = color_palette[color_index]
        
        current_color = label2color[label]

        # 1. 画目标框
        draw.rectangle(coordinates, outline=current_color, width=width)
        
        # 2. 画标签文本和背景
        text_position = (box["x1"] + 2, box["y1"] + 2)
        
        # 使用 textbbox 动态获取文本长宽 (比固定的 len(label)*8 更准确)
        if hasattr(draw, 'textbbox') and font is not None:
            text_bbox = draw.textbbox(text_position, label, font=font)
            # 稍微扩大一点背景作为 padding
            bg_bbox = [text_bbox[0] - 2, text_bbox[1] - 2, text_bbox[2] + 2, text_bbox[3] + 2]
        else:
            # 兼容老版本 PIL
            bg_bbox = [box["x1"], box["y1"], box["x1"] + len(label) * 7, box["y1"] + 15]

        # 给文字加个全实心背景，颜色就是对应的画框颜色，方便看清
        draw.rectangle(bg_bbox, fill=current_color)
        
        # 针对较亮的底色（如黄色），可以使用黑色字体；这里统一下，使用白色，对于我们的调色板可视度都不错
        draw.text(text_position, label, fill="white", font=font)
        
    return draw_img


In [34]:


# 解析坐标
w, h = image.size
boxes = parse_boxes(completion.choices[0].text, w, h)
print("Parsed boxes with labels:", boxes)

# 绘制画框
visualized_img = draw_boxes(image, boxes, width=3)


Parsed boxes with labels: [{'label': 'person', 'x1': 82.46, 'y1': 13.5, 'x2': 463.372, 'y2': 855.9}, {'label': 'person', 'x1': 246.848, 'y1': 30.6, 'x2': 486.78000000000003, 'y2': 867.6}, {'label': 'ballet heel', 'x1': 133.0, 'y1': 613.8000000000001, 'x2': 224.504, 'y2': 851.4}, {'label': 'ballet heel', 'x1': 193.648, 'y1': 603.0, 'x2': 264.404, 'y2': 855.0}, {'label': 'ballet heel', 'x1': 291.536, 'y1': 741.5999999999999, 'x2': 371.868, 'y2': 867.6}, {'label': 'ballet heel', 'x1': 375.06, 'y1': 738.9, 'x2': 447.94399999999996, 'y2': 856.8}]


In [36]:
visualized_img.show()

In [27]:
completion

Completion(id='cmpl-963528f0a85464e7', choices=[CompletionChoice(finish_reason='stop', index=0, logprobs=None, text='personballet heelbicycleNone', stop_reason=None, token_ids=[151672, 8987, 151673, 151668, 151832, 151692, 152548, 152628, 151669, 151668, 152141, 151711, 152592, 152641, 151669, 151672, 65, 7464, 34328, 151673, 151668, 151927, 152359, 152099, 152623, 151669, 151668, 152041, 152347, 152174, 152627, 151669, 151668, 152225, 152501, 152376, 152641, 151669, 151668, 152382, 152498, 152519, 152629, 151669, 151672, 65, 26165, 151673, 151668, 4064, 151669, 151645], prompt_logprobs=None, prompt_token_ids=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0